# Milestone 6 — Diffusion & Generative AI

**Task:** Generate lifestyle product images using Stable Diffusion v1.5
conditioned on a product's title and description.
**Deliverable:** demo + reflection on quality and failure modes.

In [ ]:
import os, sys, json
import pandas as pd
from PIL import Image

PROJECT_ROOT = '/content/drive/MyDrive/smart-product-intelligence'

## 1. Pre-generated examples (saved in Drive)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

m6_dir = os.path.join(PROJECT_ROOT, 'm6_generated')
generated_files = sorted(os.listdir(m6_dir))
print(f'Pre-generated images: {len(generated_files)}')
for f in generated_files:
    print(f'  - {f}')

## 2. Generation setup (re-runnable)

The full generation code is provided below. Each image takes ~8 seconds
on a T4 GPU at 25 inference steps. The cached images in `m6_generated/`
were produced with this exact code.

In [ ]:
# To regenerate (uncomment to run; requires GPU):
'''
from diffusers import StableDiffusionPipeline
import torch

pipe = StableDiffusionPipeline.from_pretrained(
    'runwayml/stable-diffusion-v1-5',
    torch_dtype=torch.float16
).to('cuda')
pipe.enable_attention_slicing()

products = pd.read_csv(os.path.join(PROJECT_ROOT, 'data', 'products.csv'))
sample = products.sample(3, random_state=42)

for _, row in sample.iterrows():
    prompt = f"professional product photograph, lifestyle setting, {row['title']}, high quality"
    image = pipe(prompt, num_inference_steps=25, width=512, height=512).images[0]
    safe_name = ''.join(c if c.isalnum() else '_' for c in row['title'])[:40]
    image.save(os.path.join(m6_dir, f'{safe_name}.png'))
'''
print('Generation code shown above. Cached images are loaded below.')

## 3. Display the generated images

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, len(generated_files), figsize=(5*len(generated_files), 5))
if len(generated_files) == 1:
    axes = [axes]

for ax, fname in zip(axes, generated_files):
    img = Image.open(os.path.join(m6_dir, fname))
    ax.imshow(img)
    ax.axis('off')
    title = fname.replace('.png', '').replace('_', ' ')[:50]
    ax.set_title(title, fontsize=10)

plt.tight_layout()
plt.show()

## 4. Failure mode analysis figure

In [ ]:
from IPython.display import Image as IPyImage
IPyImage(filename=os.path.join(PROJECT_ROOT, 'figures', '08_m6_comparison.png'))

## 5. Reflection on quality and failure modes

### Successes
- Lifestyle style, lighting, and composition are captured well.
- Effective on **generic, brandless products** (plush toys, plastic toys,
  household items).

### Failure modes (consistent across multiple prompts)

1. **Brand recognition.** Specific brands like "Funko", "Playmobil",
   "Octonauts" are NOT recognized — the model generates a generic
   substitute. Brand-specific aesthetics are completely lost.
2. **Text inside images.** Diffusion models are notoriously weak at
   rendering text. Product packaging text is gibberish.
3. **Exact counts and sizes.** Prompts like "3-inch cake topper set"
   are not respected — the model decides the size and quantity itself.
4. **Licensed characters.** Disney, Marvel, video-game characters are
   blocked by safety filters in SD 1.5, or distorted beyond recognition.

### Practical conclusion

Generation **cannot replace** the main product photo, but it **can add
value** for marketing and lifestyle banner imagery where brand-specific
detail is not required.

This honest assessment of where generative AI helps and where it
doesn't is exactly what the brief asks for: *"reason about where
generation genuinely adds value"*.

## 6. Summary

- Stable Diffusion v1.5 produces usable lifestyle imagery for generic
  products in ~8 seconds per image on a T4 GPU.
- Documented and honest failure modes: brands, text in images, exact
  counts/sizes, licensed characters.
- Generation is a marketing tool, not a product-photo replacement.